# Background Subtraction Methods

This notebook demonstrates the four background subtraction methods available in
`gs_analysis` and how to use them with a synthetic gamma spectrum created by
`gs_creator`.

| Method | `BackgroundMethod` constant | Description |
|---|---|---|
| Trapezoid (Maestro) | `TRAPEZOID` | Maestro-style linear interpolation using up to 2 channels on each side of the ROI |
| Linear interpolation | `LINEAR` | Averages several channels on each side and linearly interpolates under the peak |
| Step function | `STEP` | Constant background set to the average of both edge regions |
| Sliding window average | `SLIDING_AVERAGE` | Moving-average background estimated from channels adjacent to the ROI |

## 1. Setup – import modules

In [ ]:
import sys
import os

# Add the package root to sys.path when running from the examples/ directory
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt

import gs_creator
import gs_analysis as gs
from gs_analysis import BackgroundMethod
import gs_plotting as gsp

## 2. Create a synthetic spectrum

A flat background is created with `gs_creator.create_flat_background` and then
passed into `gs_creator.create_spectrum_from_peaks` so that the final spectrum
contains three gamma-ray photopeaks sitting on top of a constant background –
a typical scenario for demonstrating background subtraction.

Poisson noise is added via `numpy.random.poisson` to give the spectrum a
realistic statistical appearance.

In [ ]:
# Spectrum settings
NUM_BINS     = 512
ENERGY_RANGE = (0.0, 500.0)  # keV

# Three gamma lines (energies in keV, peak emission rates in counts)
PEAK_ENERGIES = [100.0, 200.0, 350.0]
PEAK_RATES    = [500,   800,   300]

# Flat background of 50 counts per bin
background = gs_creator.create_flat_background(
    num_bins=NUM_BINS,
    background_level=50,
    energy_range=ENERGY_RANGE,
    spec_name="flat background",
)

# Synthetic spectrum: peaks on top of the flat background
spec = gs_creator.create_spectrum_from_peaks(
    peak_energies=PEAK_ENERGIES,
    emission_rates=PEAK_RATES,
    num_bins=NUM_BINS,
    energy_range=ENERGY_RANGE,
    background_spectrum=background,
    fwhm_factor=0.04,  # 4 % energy resolution
    spec_name="synthetic spectrum",
)

# Add Poisson noise to give a realistic statistical appearance
np.random.seed(42)
spec.counts = np.random.poisson(spec.counts).astype(spec.counts.dtype)

# Energy axis
ebins = gs.generate_ebins(spec)

print(f"Channels     : {spec.num_channels}")
print(f"Energy range : {ebins[0]:.1f} – {ebins[-1]:.1f} keV")
print(f"Total counts : {spec.counts.sum():,}")

## 3. View the spectrum

A quick overview of the full spectrum, with the three photopeaks visible on the
flat background.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ebins, spec.counts, linewidth=0.8, color='steelblue', label='Synthetic spectrum')

for energy in PEAK_ENERGIES:
    ax.axvline(x=energy, color='crimson', linestyle='--', linewidth=0.9, alpha=0.7)

ax.set_xlabel('Energy (keV)', fontsize=12)
ax.set_ylabel('Counts', fontsize=12)
ax.set_title('Synthetic Spectrum – Three Photopeaks on Flat Background', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(ENERGY_RANGE)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

## 4. Define the ROI and convert energy to channels

Background subtraction in `gs_analysis` operates on channel indices.  The helper
below converts an energy window (in keV) to the corresponding start and end
channels so that we can analyse the **200 keV photopeak**.

In [ ]:
# Select the 200 keV photopeak and define a ±20 keV ROI
PEAK_ENERGY = 200.0  # keV
ROI_HALF_WIDTH = 20.0  # keV

c1 = int(np.searchsorted(ebins, PEAK_ENERGY - ROI_HALF_WIDTH))
c2 = int(np.searchsorted(ebins, PEAK_ENERGY + ROI_HALF_WIDTH))

print(f"ROI: channels {c1} – {c2}  ({ebins[c1]:.1f} – {ebins[c2]:.1f} keV)")

## 5. Visual comparison of background methods

`gs_plotting.plot_background_methods_comparison` produces a 2×2 grid showing
the estimated background line overlaid on the peak region for each of the four
methods.  The gross and net counts are printed in each panel's title.

In [ ]:
gsp.plot_background_methods_comparison(
    spec.counts, c1, c2,
    title=f"Background Subtraction Methods – {PEAK_ENERGY:.0f} keV Peak",
)

## 6. Numerical comparison – net counts and uncertainties

`gs_analysis.net_counts_uncertainty` returns both the background-subtracted net
counts and the one-sigma Poisson uncertainty for a chosen `BackgroundMethod`.
Running all four methods makes it easy to compare their estimates side by side.

In [ ]:
method_names = {
    BackgroundMethod.TRAPEZOID:      "Trapezoid (Maestro)",
    BackgroundMethod.LINEAR:         "Linear interpolation",
    BackgroundMethod.STEP:           "Step function",
    BackgroundMethod.SLIDING_AVERAGE: "Sliding window average",
}

gross = gs.gross_count(spec.counts, c1, c2)
print(f"Gross counts (ROI): {gross}")
print()
print(f"{'Method':<28} {'Background':>12} {'Net counts':>12} {'Uncertainty (1σ)':>18}")
print("-" * 74)

for method, name in method_names.items():
    bg  = gs.calc_bg(spec.counts, c1, c2, m=method)
    net, unc = gs.net_counts_uncertainty(spec.counts, c1, c2, m=method)
    print(f"{name:<28} {bg:>12.1f} {net:>12.1f} {unc:>18.1f}")

## 7. Sensitivity to background model choice

`gs_analysis.peak_area_with_background_sensitivity` runs all four methods and
reports the mean net counts together with the standard deviation across methods.
A large standard deviation indicates that the result is strongly sensitive to the
chosen background model and should be interpreted with care.

In [ ]:
mean_net, std_net, per_method = gs.peak_area_with_background_sensitivity(
    spec.counts, c1, c2
)

print(f"Mean net counts  : {mean_net:.1f}")
print(f"Std dev (methods): {std_net:.1f}  ({100 * std_net / mean_net:.1f} %)")
print()
print("Per-method breakdown:")
for method_name, net in per_method.items():
    print(f"  {method_name:<20}: {net:.1f}")

## Summary

This notebook demonstrated background subtraction in `gs_analysis`:

1. **Create** a synthetic spectrum with `gs_creator.create_spectrum_from_peaks`,
   adding a flat background via `gs_creator.create_flat_background`.
2. **Define a ROI** around a photopeak by converting an energy window to channel
   indices.
3. **Visualise** all four background methods at once with
   `gs_plotting.plot_background_methods_comparison`.
4. **Quantify** the background, net counts, and statistical uncertainty for each
   method using `gs_analysis.calc_bg` and `gs_analysis.net_counts_uncertainty`.
5. **Assess** the sensitivity of the result to the choice of background model with
   `gs_analysis.peak_area_with_background_sensitivity`.

For a flat background the four methods give very consistent net-count estimates;
differences become larger when the background has a non-trivial shape under the
peak.